# Análise de features — base numérica (ENEM 2023, Matemática)

Este notebook trabalha **sobre a base numérica** gerada por `src/higienizacao.py` (138 colunas, 33 features codificadas pelo dicionário) e responde a três perguntas:

1. **Features derivadas:** quais combinações simples (renda per capita, escolaridade máxima dos responsáveis, acesso digital, adensamento e interação renda × escolaridade) podemos criar a partir das colunas numéricas? Todas são calculadas linha a linha, **sem aprender nada com os dados**, então não geram vazamento.
2. **Redundância:** quais features numéricas são muito parecidas entre si (correlação de Spearman e `FeatureAgglomeration`)?
3. **Relação com a nota e valor preditivo:** o que o `SelectKBest(f_regression)` aponta e, principalmente, se as derivadas e a seleção **melhoram o erro** na validação cruzada (mesmas dobras, `KFold(3, shuffle=True, random_state=42)`).

**Só o conjunto de desenvolvimento é lido; o teste não é carregado.** O ranking e as correlações usam uma amostra de 100 mil participantes do desenvolvimento apenas para diagnóstico; a decisão se apoia na validação cruzada com todo o desenvolvimento. Códigos categóricos (dummies) não entram em correlação de Pearson/Spearman: ela só vale para colunas ordinais ou contínuas.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from scipy import sparse

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / "dados").exists() and (RAIZ_PROJETO.parent / "dados").exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent
sys.path.insert(0, str(RAIZ_PROJETO / "src"))

from modelagem_acompanhamento import carregar_desenvolvimento_esparso
from registro_experimentos import linha_de_cv, registrar

# Desenvolvimento completo em formato esparso (o arquivo de teste não é referenciado).
X_dev, y_dev, NOMES = carregar_desenvolvimento_esparso()
COL = {nome: i for i, nome in enumerate(NOMES)}
print(f"Desenvolvimento: {X_dev.shape[0]:,} participantes | {X_dev.shape[1]} features numéricas")

  blocos lidos: 5 | linhas: 500,000


  blocos lidos: 10 | linhas: 1,000,000


  blocos lidos: 15 | linhas: 1,500,000


  blocos lidos: 20 | linhas: 2,000,000


Desenvolvimento: 2,153,941 participantes | 138 features numéricas


## 1. Features derivadas

Cada feature tem uma hipótese e um cuidado. Denominador inválido (menor que 1 ou ausente) vira **ausente**, e não zero; a imputação pela mediana fica no `Pipeline`, ajustada só no treino de cada dobra.

| Feature | Fórmula | Hipótese | Cuidado |
|---|---|---|---|
| `RENDA_PER_CAPITA` | renda estimada (SM) ÷ moradores | Renda por morador descreve melhor a condição material que a renda total. | A renda é uma faixa, então o erro da aproximação se multiplica. |
| `ESC_MAX_RESP` | maior nível de escolaridade entre os responsáveis | O maior nível do domicílio resume o capital educacional. | Se só um responsável é conhecido, o outro é ignorado. |
| `INDICE_DIGITAL` | computador em casa (0/1) + internet (0/1) | Computador e internet favorecem o estudo. | Resposta ausente conta como "não tem". |
| `MORADORES_POR_QUARTO` | moradores ÷ quartos | Domicílio mais adensado dificulta o estudo. | Quartos igual a 0 ou ausente vira ausente. |
| `RENDA_X_ESC_MAX` | renda estimada × escolaridade máxima | Renda e escolaridade se reforçam. | Muito correlacionada com as colunas que a originam. |

In [2]:
def coluna(X, nome):
    """Uma coluna da matriz esparsa como vetor denso (NaN preservado)."""
    return X[:, COL[nome]].toarray().ravel().astype("float32")


def razao(numerador, denominador):
    """Divisão com denominador inválido (menor que 1 ou ausente) explicitado como ausente."""
    denominador = np.where(denominador >= 1, denominador, np.nan)
    return (numerador / denominador).astype("float32")


def calcular_derivadas(X):
    """Features derivadas a partir das colunas numéricas; nada é aprendido dos dados."""
    renda = coluna(X, "Q006_RENDA_SM_ESTIMADA")
    moradores = coluna(X, "Q005_MORADORES")
    quartos = coluna(X, "Q009_CONTAGEM_ESTIMADA")
    esc_max = np.fmax(coluna(X, "Q001_NIVEL_ESCOLARIDADE"), coluna(X, "Q002_NIVEL_ESCOLARIDADE"))
    tem_computador = np.nan_to_num(coluna(X, "Q024_CONTAGEM_ESTIMADA")) > 0
    tem_internet = np.nan_to_num(coluna(X, "Q025_BINARIA")) == 1
    return pd.DataFrame({
        "RENDA_PER_CAPITA": razao(renda, moradores),
        "ESC_MAX_RESP": esc_max,
        "INDICE_DIGITAL": (tem_computador.astype("float32") + tem_internet.astype("float32")),
        "MORADORES_POR_QUARTO": razao(moradores, quartos),
        "RENDA_X_ESC_MAX": (renda * esc_max).astype("float32"),
    })


derivadas_dev = calcular_derivadas(X_dev)
resumo_derivadas = derivadas_dev.describe().T[["min", "50%", "max"]].rename(columns={"50%": "mediana"})
resumo_derivadas["ausentes_%"] = (derivadas_dev.isna().mean() * 100).round(2)
display(resumo_derivadas)

,min,mediana,max,ausentes_%
RENDA_PER_CAPITA,0.00,0.4375,20.0,0.00
ESC_MAX_RESP,0.00,4.0000,6.0,2.01
INDICE_DIGITAL,0.00,2.0000,2.0,0.00
MORADORES_POR_QUARTO,0.25,1.5000,20.0,0.72
RENDA_X_ESC_MAX,0.00,5.2500,120.0,2.01


## 2. Redundância entre features numéricas

A correlação de Spearman é calculada só entre as colunas **ordinais ou contínuas** (escolaridade, renda, moradores, contagens, binárias e as derivadas), em uma amostra de 100 mil participantes do desenvolvimento. Pares com |ρ| ≥ 0,85 são **candidatos a revisão**, não uma ordem para apagar colunas.

In [3]:
LIMITE_CORRELACAO = 0.85
ORDINAIS = (
    ["Q001_NIVEL_ESCOLARIDADE", "Q002_NIVEL_ESCOLARIDADE", "Q005_MORADORES",
     "Q006_RENDA_SM_ESTIMADA", "Q007_DIAS_DOMESTICA_ESTIMADOS"]
    + [n for n in NOMES if n.endswith("_CONTAGEM_ESTIMADA") or n.endswith("_BINARIA")]
)

rng = np.random.RandomState(42)
idx_amostra = np.sort(rng.choice(X_dev.shape[0], size=100_000, replace=False))
X_amostra = X_dev[idx_amostra]
y_amostra = y_dev[idx_amostra]
derivadas_amostra = derivadas_dev.iloc[idx_amostra].reset_index(drop=True)

base_ordinal = pd.DataFrame(X_amostra[:, [COL[n] for n in ORDINAIS]].toarray(), columns=ORDINAIS)
matriz_numerica = pd.concat([base_ordinal, derivadas_amostra], axis=1)
matriz_numerica = matriz_numerica.loc[:, matriz_numerica.std() > 0]   # constante não tem correlação
correlacao = matriz_numerica.corr(method="spearman", min_periods=500)

mascara = np.triu(np.ones(correlacao.shape, dtype=bool), k=1)
pares = (correlacao.where(mascara).stack().rename("rho_spearman").reset_index()
         .rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
pares["|rho|"] = pares["rho_spearman"].abs()
pares_altos = pares.loc[pares["|rho|"] >= LIMITE_CORRELACAO].sort_values("|rho|", ascending=False, ignore_index=True)
print(f"Amostra diagnóstica: {len(matriz_numerica):,} participantes do desenvolvimento | "
      f"{matriz_numerica.shape[1]} colunas | pares com |rho| >= {LIMITE_CORRELACAO}: {len(pares_altos)}")
display(pares_altos)

fig_corr = px.imshow(correlacao, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto",
                     title="Spearman entre features ordinais/contínuas e derivadas (amostra do desenvolvimento)")
fig_corr.update_layout(template="plotly_white", height=800)
fig_corr.show()

Amostra diagnóstica: 100,000 participantes do desenvolvimento | 28 colunas | pares com |rho| >= 0.85: 5


,feature_a,feature_b,rho_spearman,|rho|
0,Q006_RENDA_SM_ESTIMADA,RENDA_X_ESC_MAX,0.950354,0.950354
1,Q006_RENDA_SM_ESTIMADA,RENDA_PER_CAPITA,0.943309,0.943309
2,Q002_NIVEL_ESCOLARIDADE,ESC_MAX_RESP,0.912788,0.912788
3,Q024_CONTAGEM_ESTIMADA,INDICE_DIGITAL,0.909729,0.909729
4,RENDA_PER_CAPITA,RENDA_X_ESC_MAX,0.898241,0.898241


### Sugestão automática de grupos

`FeatureAgglomeration` agrupa colunas parecidas (distância `1 − |correlação|` abaixo do limiar). O resultado é uma **lista de candidatos à fusão**, não uma fusão aplicada.

In [4]:
from sklearn.cluster import FeatureAgglomeration
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_padronizada = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(matriz_numerica))
aglomerador = FeatureAgglomeration(n_clusters=None, metric="correlation", linkage="average",
                                   distance_threshold=1 - LIMITE_CORRELACAO).fit(X_padronizada)
grupos = (pd.DataFrame({"feature": matriz_numerica.columns, "grupo": aglomerador.labels_})
          .groupby("grupo")["feature"].agg(list).loc[lambda g: g.str.len() > 1]
          .reset_index(name="features_candidatas_a_fusao"))
if grupos.empty:
    print("Nenhum grupo ultrapassou o limiar; não há sugestão automática de fusão neste corte.")
else:
    display(grupos)

,grupo,features_candidatas_a_fusao
0,0,"[Q006_RENDA_SM_ESTIMADA, RENDA_PER_CAPITA, REN..."
1,1,"[Q002_NIVEL_ESCOLARIDADE, ESC_MAX_RESP]"


## 3. Relação com a nota: `SelectKBest(f_regression)`

O F-score univariado mede a **associação linear** de cada coluna com a nota (não causalidade, e não captura interações). Como as dummies vêm de uma feature original, o ranking é agregado pela feature de origem (maior F-score entre suas colunas). É um diagnóstico em amostra; o efeito real é medido na seção 4.

In [5]:
import re
from sklearn.feature_selection import SelectKBest, f_regression

ORIGENS_PERFIL = ["TP_FAIXA_ETARIA", "TP_SEXO", "TP_ESTADO_CIVIL", "TP_COR_RACA",
                  "TP_ST_CONCLUSAO", "TP_ESCOLA", "TP_ENSINO", "IN_TREINEIRO"]


def feature_de_origem(nome):
    if nome in derivadas_dev.columns:
        return nome
    if re.match(r"Q\d{3}", nome):
        return nome[:4]
    return max((o for o in ORIGENS_PERFIL if nome.startswith(o)), key=len, default=nome)


nomes_todos = NOMES + list(derivadas_dev.columns)
X_amostra_densa = np.hstack([X_amostra.toarray(), derivadas_amostra.to_numpy(dtype="float32")])
X_amostra_imputada = SimpleImputer(strategy="median", keep_empty_features=True).fit_transform(X_amostra_densa)
seletor = SelectKBest(score_func=f_regression, k="all").fit(X_amostra_imputada, y_amostra)

ranking_colunas = pd.DataFrame({"coluna": nomes_todos, "f_score": np.nan_to_num(seletor.scores_)})
ranking_colunas["feature_origem"] = ranking_colunas["coluna"].map(feature_de_origem)
ranking = (ranking_colunas.groupby("feature_origem", as_index=False)
           .agg(f_score_maximo=("f_score", "max"), colunas=("coluna", "count"))
           .sort_values("f_score_maximo", ascending=False, ignore_index=True))
ranking["derivada"] = ranking["feature_origem"].isin(derivadas_dev.columns)
display(ranking.head(25))

fig_rank = px.bar(ranking.head(20).iloc[::-1], x="f_score_maximo", y="feature_origem", orientation="h",
                  color="derivada", color_discrete_map={True: "#d62728", False: "#1f77b4"},
                  title="Top 20 features por F-score (amostra do desenvolvimento; vermelho = derivada)",
                  labels={"f_score_maximo": "F-score (f_regression)", "feature_origem": ""})
fig_rank.update_layout(template="plotly_white", height=600)
fig_rank.show()

,feature_origem,f_score_maximo,colunas,derivada
0,Q024,19743.902750,3,False
1,Q006,19329.834882,3,False
2,RENDA_X_ESC_MAX,18979.185496,1,True
3,RENDA_PER_CAPITA,16916.793796,1,True
4,INDICE_DIGITAL,14970.371412,1,True
5,ESC_MAX_RESP,14252.807275,1,True
6,Q008,13313.696624,3,False
7,Q002,12458.707156,4,False
8,Q010,12423.571620,3,False
9,Q001,11918.702242,4,False


## 4. Comparação com validação cruzada

Três representações, **mesmo regressor** (`Ridge(alpha=1)`), **mesmas dobras** e o desenvolvimento completo:

- **A. Originais:** as 138 colunas codificadas.
- **B. Originais + derivadas:** as 138 colunas mais as 5 derivadas (calculadas antes, pois não aprendem nada dos dados).
- **C. B + `SelectKBest(k=50)`:** a seleção fica dentro do `Pipeline` e é reajustada no treino de cada dobra.

Imputação pela mediana e padronização também ficam no `Pipeline`. Os resultados são gravados em `resultados/experimentos.csv`.

In [6]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

K_SELECAO = 50
cv = KFold(n_splits=3, shuffle=True, random_state=42)
scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}


def criar_pipeline(k=None):
    etapas = [("imputacao", SimpleImputer(strategy="median", keep_empty_features=True)),
              ("padronizacao", StandardScaler(with_mean=False))]
    if k:
        etapas.append(("selecao", SelectKBest(score_func=f_regression, k=k)))
    etapas.append(("modelo", Ridge(alpha=1.0, solver="lsqr")))
    return Pipeline(etapas)


X_com_derivadas = sparse.hstack(
    [X_dev, sparse.csr_matrix(derivadas_dev.to_numpy(dtype="float32"))], format="csr", dtype=np.float32)

experimentos = {
    "A. Originais (138)": (X_dev, None),
    "B. Originais + derivadas (143)": (X_com_derivadas, None),
    f"C. B + SelectKBest (k={K_SELECAO})": (X_com_derivadas, K_SELECAO),
}
resultados_cv = {}
for nome, (X_exp, k) in experimentos.items():
    print(f"Validação cruzada: {nome}")
    resultados_cv[nome] = cross_validate(criar_pipeline(k), X_exp, y_dev, cv=cv, scoring=scoring,
                                         n_jobs=1, error_score="raise")

linhas = []
for nome, r in resultados_cv.items():
    linhas.append({
        "representação": nome,
        "RMSE": f"{-r['test_rmse'].mean():.3f} ± {r['test_rmse'].std(ddof=1):.3f}",
        "MAE": f"{-r['test_mae'].mean():.3f} ± {r['test_mae'].std(ddof=1):.3f}",
        "R²": f"{r['test_r2'].mean():.4f} ± {r['test_r2'].std(ddof=1):.4f}",
        "tempo_ajuste_s": round(float(r["fit_time"].sum()), 1),
    })
tabela_cv = pd.DataFrame(linhas)
display(tabela_cv)

base = resultados_cv["A. Originais (138)"]["test_rmse"]
for nome, r in resultados_cv.items():
    diferenca_por_dobra = (-r["test_rmse"]) - (-base)   # negativo = melhor que as originais
    print(f"{nome}: diferença de RMSE vs A por dobra = {np.round(diferenca_por_dobra, 3)}")

Validação cruzada: A. Originais (138)


Validação cruzada: B. Originais + derivadas (143)


Validação cruzada: C. B + SelectKBest (k=50)


,representação,RMSE,MAE,R²,tempo_ajuste_s
0,A. Originais (138),110.647 ± 0.175,86.601 ± 0.057,0.2935 ± 0.0006,4.1
1,B. Originais + derivadas (143),110.447 ± 0.179,86.383 ± 0.057,0.2960 ± 0.0007,6.1
2,C. B + SelectKBest (k=50),111.284 ± 0.190,87.083 ± 0.049,0.2853 ± 0.0007,5.6


A. Originais (138): diferença de RMSE vs A por dobra = [0. 0. 0.]
B. Originais + derivadas (143): diferença de RMSE vs A por dobra = [-0.198 -0.205 -0.194]
C. B + SelectKBest (k=50): diferença de RMSE vs A por dobra = [0.604 0.655 0.652]


In [7]:
preparacao = "imputação pela mediana + padronização esparsa dentro do Pipeline, por dobra"
registrar([
    linha_de_cv("feat_A_originais_ridge", "Referência: 33 features codificadas (138 colunas)",
                "138 colunas numéricas (33 features originais codificadas)", preparacao, "Ridge",
                "alpha=1.0; solver=lsqr", resultados_cv["A. Originais (138)"],
                "Referência para comparar as derivadas e a seleção."),
    linha_de_cv("feat_B_derivadas_ridge", "As 5 features derivadas melhoram o erro em relação às originais",
                "138 colunas + RENDA_PER_CAPITA, ESC_MAX_RESP, INDICE_DIGITAL, MORADORES_POR_QUARTO, RENDA_X_ESC_MAX",
                preparacao, "Ridge", "alpha=1.0; solver=lsqr", resultados_cv["B. Originais + derivadas (143)"],
                "RMSE cerca de 0,2 ponto menor que as originais, nas três dobras (ganho pequeno, ~0,18%); as derivadas são parcialmente redundantes com as colunas de origem."),
    linha_de_cv(f"feat_C_selectkbest{K_SELECAO}_ridge", f"Selecionar {K_SELECAO} colunas mantém o desempenho com menos features",
                f"143 colunas, depois SelectKBest(f_regression, k={K_SELECAO}) dentro do Pipeline", preparacao, "Ridge",
                f"alpha=1.0; solver=lsqr; k={K_SELECAO}", resultados_cv[f"C. B + SelectKBest (k={K_SELECAO})"],
                "SelectKBest(k=50) piorou o RMSE em ~0,6 ponto frente às originais, nas três dobras; com esse filtro univariado a hipótese de manter o desempenho com menos features não se confirmou."),
], pasta_saida=RAIZ_PROJETO / "resultados")
print("Experimentos registrados em resultados/experimentos.csv")

Experimentos registrados em resultados/experimentos.csv


## 4.1 Fusão dos grupos candidatos

Na seção 2 o `FeatureAgglomeration` sugeriu dois grupos de colunas muito correlacionadas (|ρ| ≥ 0,85). Aqui a fusão é **aplicada de fato** e comparada com a representação B, nas mesmas dobras:

- **`FUSAO_RENDA`** = renda estimada (`Q006`), renda per capita e interação renda × escolaridade.
- **`FUSAO_ESCOLARIDADE`** = escolaridade da mãe (`Q002`) e escolaridade máxima dos responsáveis.

Cada grupo vira **uma coluna: a média das colunas padronizadas**. A média e o desvio de cada coluna são aprendidos só no treino, dentro do `Pipeline` (`src/fusao_features.py`), então não há vazamento entre dobras. As 5 colunas de origem saem e entram as 2 fundidas (143 → 140 colunas). O par computador (`Q024`) × índice digital passa de 0,85, mas não formou grupo, e por isso **não é fundido**.

**D. B com grupos fundidos** usa o mesmo `Ridge(alpha=1)`. A pergunta é se juntar as colunas redundantes mantém o erro com menos colunas, ou se a fusão joga fora informação (por exemplo, a interação renda × escolaridade).

In [ ]:
from fusao_features import GRUPOS_FUSAO, FusaoPorGrupos, indices_dos_grupos, nomes_apos_fusao

nomes_B = NOMES + list(derivadas_dev.columns)
nomes_D = nomes_apos_fusao(nomes_B)
grupos_idx = indices_dos_grupos(nomes_B)

tabela_grupos = pd.DataFrame([
    {"feature_fundida": grupo, "colunas_de_origem": ", ".join(membros), "n_origem": len(membros)}
    for grupo, membros in GRUPOS_FUSAO.items()
])
display(tabela_grupos)
n_origem = int(tabela_grupos["n_origem"].sum())
print(f"B: {len(nomes_B)} colunas -> D: {len(nomes_D)} colunas "
      f"({n_origem} colunas de origem trocadas por {len(GRUPOS_FUSAO)} fundidas)")


def criar_pipeline_fusao():
    return Pipeline([
        ("imputacao", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("fusao", FusaoPorGrupos(grupos_idx)),
        ("padronizacao", StandardScaler(with_mean=False)),
        ("modelo", Ridge(alpha=1.0, solver="lsqr")),
    ])


NOME_D = f"D. B com grupos fundidos ({len(nomes_D)})"
print(f"Validação cruzada: {NOME_D}")
resultados_cv[NOME_D] = cross_validate(criar_pipeline_fusao(), X_com_derivadas, y_dev, cv=cv, scoring=scoring,
                                       n_jobs=1, error_score="raise")

linhas_d = []
for nome in ("A. Originais (138)", "B. Originais + derivadas (143)", NOME_D):
    r = resultados_cv[nome]
    linhas_d.append({
        "representação": nome,
        "RMSE": f"{-r['test_rmse'].mean():.3f} ± {r['test_rmse'].std(ddof=1):.3f}",
        "MAE": f"{-r['test_mae'].mean():.3f} ± {r['test_mae'].std(ddof=1):.3f}",
        "R²": f"{r['test_r2'].mean():.4f} ± {r['test_r2'].std(ddof=1):.4f}",
        "tempo_ajuste_s": round(float(r["fit_time"].sum()), 1),
    })
display(pd.DataFrame(linhas_d))

rmse_A = -resultados_cv["A. Originais (138)"]["test_rmse"]
rmse_B, rmse_D = -resultados_cv["B. Originais + derivadas (143)"]["test_rmse"], -resultados_cv[NOME_D]["test_rmse"]
dif_D_B = rmse_D - rmse_B   # positivo = D pior que B
dif_A_D = rmse_A - rmse_D   # positivo = D melhor que A
print(f"Diferença de RMSE D - B por dobra = {np.round(dif_D_B, 3)} (positivo = a fusão piorou)")

if (dif_D_B > 0).all():
    leitura = (f"Fundir os grupos piorou o RMSE em {dif_D_B.mean():.2f} ponto frente a B, nas três dobras, mas ainda ficou "
               f"{dif_A_D.mean():.2f} ponto abaixo de A (originais); a fusão perdeu parte do ganho das derivadas, "
               f"com {len(nomes_B) - len(nomes_D)} colunas a menos.")
elif (dif_D_B < 0).all():
    leitura = (f"Fundir os grupos melhorou o RMSE em {-dif_D_B.mean():.2f} ponto frente a B, nas três dobras, "
               f"com {len(nomes_B) - len(nomes_D)} colunas a menos.")
else:
    leitura = (f"Sem diferença consistente entre as dobras (média de {dif_D_B.mean():+.2f} ponto frente a B), "
               f"mas com {len(nomes_B) - len(nomes_D)} colunas a menos.")
print(leitura)

registrar([
    linha_de_cv("feat_D_fusao_grupos_ridge",
                "Fundir os grupos redundantes (renda e escolaridade) mantém o erro com menos colunas",
                "143 colunas, com 5 trocadas por FUSAO_RENDA (Q006, RENDA_PER_CAPITA, RENDA_X_ESC_MAX) e "
                "FUSAO_ESCOLARIDADE (Q002, ESC_MAX_RESP), média dos z-scores ajustados no treino",
                "imputação pela mediana + fusão por grupos + padronização esparsa dentro do Pipeline, por dobra",
                "Ridge", "alpha=1.0; solver=lsqr", resultados_cv[NOME_D], leitura),
], pasta_saida=RAIZ_PROJETO / "resultados")
print("Experimento D registrado em resultados/experimentos.csv")

,feature_fundida,colunas_de_origem,n_origem
0,FUSAO_RENDA,"Q006_RENDA_SM_ESTIMADA, RENDA_PER_CAPITA, REND...",3
1,FUSAO_ESCOLARIDADE,"Q002_NIVEL_ESCOLARIDADE, ESC_MAX_RESP",2


B: 143 colunas -> D: 140 colunas (5 colunas de origem trocadas por 2 fundidas)
Validação cruzada: D. B com grupos fundidos (140)


,representação,RMSE,MAE,R²,tempo_ajuste_s
0,A. Originais (138),110.647 ± 0.175,86.601 ± 0.057,0.2935 ± 0.0006,4.7
1,B. Originais + derivadas (143),110.447 ± 0.179,86.383 ± 0.057,0.2960 ± 0.0007,6.8
2,D. B com grupos fundidos (140),110.553 ± 0.172,86.503 ± 0.050,0.2947 ± 0.0006,6.3


Diferença de RMSE D - B por dobra = [0.107 0.112 0.097] (positivo = a fusão piorou)
Fundir os grupos piorou o RMSE em 0.11 ponto frente a B, nas três dobras, mas ainda ficou 0.09 ponto abaixo de A (originais); a fusão perdeu parte do ganho das derivadas, com 3 colunas a menos.
Experimento D registrado em resultados/experimentos.csv


## 5. Conclusões

**Features derivadas (B × A).** Acrescentar as 5 derivadas baixou o RMSE de **110,65 para 110,45** (R² de 0,2935 para 0,2960). O ganho é **pequeno, cerca de 0,2 ponto de nota ou 0,18%**, mas aparece **nas três dobras** (−0,198, −0,205 e −0,194), e o desvio entre dobras de cada representação é 0,18. É um ganho consistente, porém modesto, e vale para um modelo **linear**. Em modelos de árvore, que já aprendem interações e razões, o efeito deve ser menor. Isso será verificado quando esses modelos forem treinados.

**Seleção de atributos (C × B).** Reduzir de 143 para 50 colunas com `SelectKBest(k=50)` **piorou** o erro (RMSE 111,28, cerca de 0,6 ponto acima das originais, também nas três dobras). Com esse filtro univariado, a hipótese "manter o desempenho com menos features" **não se confirmou**: o F-score mede associação individual e descarta colunas que só ajudam em conjunto. Um `k` maior ou outro método (por exemplo, importância por árvore) ainda podem ser testados.

**Redundância.** Cinco pares têm |ρ| ≥ 0,85: renda estimada com `RENDA_X_ESC_MAX` e com `RENDA_PER_CAPITA`, escolaridade da mãe com `ESC_MAX_RESP`, computador com `INDICE_DIGITAL` e renda per capita com a interação. O `FeatureAgglomeration` sugere dois grupos: o da renda e o da escolaridade. As derivadas são, portanto, **parcialmente redundantes** com as colunas que as originam, o que explica o ganho pequeno. Na seção 4.1 os dois grupos foram **fundidos** (143 → 140 colunas, média das colunas padronizadas) e comparados com B: o RMSE ficou em 110,553, cerca de 0,11 ponto **pior** que B e 0,09 ponto melhor que A, nas três dobras. Ou seja, a fusão simplificou a representação, mas perdeu parte do ganho das derivadas.

**Ranking de associação.** No diagnóstico em amostra, computador (`Q024`) e renda (`Q006`) lideram, seguidos de `RENDA_X_ESC_MAX`, `RENDA_PER_CAPITA`, `INDICE_DIGITAL` e `ESC_MAX_RESP`. É associação linear com a nota, e não causalidade.

**Limitações.** Todos os resultados vêm do desenvolvimento (validação cruzada), com `Ridge(alpha=1)` sem busca de hiperparâmetros; o teste não foi usado. As conclusões devem ser revistas com os demais regressores.

## 6. Próximos passos

1. Repetir a comparação A × B com modelos de árvore (Random Forest ou Gradient Boosting) para ver se o ganho das derivadas se mantém.
2. Testar outro método de seleção ou um `k` maior, já que `k=50` piorou o erro.
3. Retirar um bloco por vez (renda, escolaridade, bens, acesso digital) e medir a variação do RMSE, que é uma das hipóteses de aprofundamento.

## 7. Painel interativo das features

Ligue e desligue as **features derivadas**, escolha o `k` do `SelectKBest` (0 = sem seleção) e o `alpha` do Ridge. Ao soltar o controle, o modelo é treinado de novo (treino em duas dobras do desenvolvimento, validação na terceira; **o teste não é lido**) e os gráficos de previsão e erro são atualizados, com o histórico das tentativas.

É uma exploração rápida de uma única dobra: a comparação oficial é a da seção 4, com validação cruzada completa. Rode a célula abaixo com o kernel do `.venv`; o controle só aparece com o kernel ativo.

In [ ]:
sys.path.insert(0, str(RAIZ_PROJETO / "src"))
%matplotlib inline
from ajuste_interativo import painel_features, separar_dobra

# Mesma matriz da seção 4 (138 originais + 5 derivadas); a dobra é feita uma vez e reaproveitada.
dobra_features = separar_dobra(X_com_derivadas, y_dev)
painel_features(dobra_features, n_colunas_originais=X_dev.shape[1])